# Part 12: MLOps & Production ML

---
## 12.1 MLOps Overview

**What is MLOps?**

Set of practices to deploy and maintain ML models in production reliably and efficiently.

**Key Principles:**
- **Automation:** CI/CD pipelines for models
- **Versioning:** Track data, code, models
- **Monitoring:** Performance, drift, health
- **Reproducibility:** Experiments and results
- **Collaboration:** Data scientists, engineers, ops

**MLOps Lifecycle:**
1. Data Engineering
2. Model Development
3. Model Training
4. Model Evaluation
5. Model Deployment
6. Model Monitoring
7. Model Retraining

---
## 12.2 Data Layer

**Components:**
- Data ingestion
- Data validation
- Data versioning
- Feature engineering
- Feature store

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import hashlib
import json
from datetime import datetime

# Data Versioning
class DataVersionControl:
    """Track data versions and lineage"""

    def __init__(self):
        self.versions = {}

    def compute_hash(self, data):
        """Compute data hash for versioning"""
        return hashlib.md5(pd.util.hash_pandas_object(data).values).hexdigest()

    def register_dataset(self, name, data, metadata=None):
        """Register dataset version"""
        version_hash = self.compute_hash(data)

        version_info = {
            'hash': version_hash,
            'shape': data.shape,
            'columns': list(data.columns),
            'timestamp': datetime.now().isoformat(),
            'metadata': metadata or {}
        }

        if name not in self.versions:
            self.versions[name] = []

        self.versions[name].append(version_info)
        print(f"Dataset '{name}' registered: {version_hash[:8]}")
        return version_hash

    def get_versions(self, name):
        """Get all versions of a dataset"""
        return pd.DataFrame(self.versions.get(name, []))

# Example usage
dvc = DataVersionControl()

# Generate sample data
np.random.seed(42)
df = pd.DataFrame({
    'feature1': np.random.randn(100),
    'feature2': np.random.randn(100),
    'target': np.random.randint(0, 2, 100)
})

# Register version
v1_hash = dvc.register_dataset('training_data', df, {'source': 'production'})

# Modify data
df['feature3'] = np.random.randn(100)
v2_hash = dvc.register_dataset('training_data', df, {'source': 'production', 'change': 'added feature3'})

# View versions
print("\nDataset versions:")
print(dvc.get_versions('training_data'))

### Data Validation

**Purpose:** Ensure data quality before training/serving

In [ ]:
class DataValidator:
    """Validate data schema and quality"""

    def __init__(self, schema):
        self.schema = schema

    def validate(self, data):
        """Validate data against schema"""
        errors = []

        # Check columns
        expected_cols = set(self.schema.keys())
        actual_cols = set(data.columns)

        missing = expected_cols - actual_cols
        if missing:
            errors.append(f"Missing columns: {missing}")

        extra = actual_cols - expected_cols
        if extra:
            errors.append(f"Extra columns: {extra}")

        # Check data types and ranges
        for col, rules in self.schema.items():
            if col not in data.columns:
                continue

            # Type check
            if 'dtype' in rules:
                if data[col].dtype != rules['dtype']:
                    errors.append(f"Column '{col}': expected {rules['dtype']}, got {data[col].dtype}")

            # Range check
            if 'min' in rules:
                if data[col].min() < rules['min']:
                    errors.append(f"Column '{col}': values below min {rules['min']}")

            if 'max' in rules:
                if data[col].max() > rules['max']:
                    errors.append(f"Column '{col}': values above max {rules['max']}")

            # Null check
            if rules.get('nullable', False) == False:
                if data[col].isnull().any():
                    errors.append(f"Column '{col}': contains null values")

        return len(errors) == 0, errors

# Define schema
schema = {
    'feature1': {'dtype': 'float64', 'min': -5, 'max': 5, 'nullable': False},
    'feature2': {'dtype': 'float64', 'min': -5, 'max': 5, 'nullable': False},
    'target': {'dtype': 'int64', 'min': 0, 'max': 1, 'nullable': False}
}

validator = DataValidator(schema)

# Validate
is_valid, errors = validator.validate(df[['feature1', 'feature2', 'target']])
print(f"Valid: {is_valid}")
if errors:
    print("Errors:")
    for error in errors:
        print(f"  - {error}")

---
## 12.3 Model Development & Training

**Best Practices:**
- Experiment tracking
- Hyperparameter tuning
- Model versioning
- Reproducibility

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle
import os

class ExperimentTracker:
    """Track ML experiments"""

    def __init__(self):
        self.experiments = []

    def log_experiment(self, name, params, metrics, artifacts=None):
        """Log experiment details"""
        experiment = {
            'name': name,
            'timestamp': datetime.now().isoformat(),
            'params': params,
            'metrics': metrics,
            'artifacts': artifacts or {}
        }
        self.experiments.append(experiment)
        print(f"Experiment '{name}' logged")
        return len(self.experiments) - 1

    def get_experiments(self):
        """Get all experiments as DataFrame"""
        if not self.experiments:
            return pd.DataFrame()

        records = []
        for exp in self.experiments:
            record = {
                'name': exp['name'],
                'timestamp': exp['timestamp'],
                **exp['params'],
                **exp['metrics']
            }
            records.append(record)

        return pd.DataFrame(records)

    def get_best_experiment(self, metric='accuracy', mode='max'):
        """Get best experiment by metric"""
        df = self.get_experiments()
        if df.empty:
            return None

        if mode == 'max':
            idx = df[metric].idxmax()
        else:
            idx = df[metric].idxmin()

        return self.experiments[idx]

# Example: Train models with experiment tracking
tracker = ExperimentTracker()

# Prepare data
X = df[['feature1', 'feature2', 'feature3']]
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Experiment 1: Random Forest with default params
model1 = RandomForestClassifier(n_estimators=10, random_state=42)
model1.fit(X_train, y_train)
y_pred1 = model1.predict(X_test)

tracker.log_experiment(
    name='rf_baseline',
    params={'n_estimators': 10, 'max_depth': None},
    metrics={
        'accuracy': accuracy_score(y_test, y_pred1),
        'precision': precision_score(y_test, y_pred1, zero_division=0),
        'recall': recall_score(y_test, y_pred1, zero_division=0),
        'f1': f1_score(y_test, y_pred1, zero_division=0)
    }
)

# Experiment 2: Random Forest with tuned params
model2 = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)

tracker.log_experiment(
    name='rf_tuned',
    params={'n_estimators': 50, 'max_depth': 5},
    metrics={
        'accuracy': accuracy_score(y_test, y_pred2),
        'precision': precision_score(y_test, y_pred2, zero_division=0),
        'recall': recall_score(y_test, y_pred2, zero_division=0),
        'f1': f1_score(y_test, y_pred2, zero_division=0)
    }
)

# View experiments
print("\nAll experiments:")
print(tracker.get_experiments())

# Get best experiment
best = tracker.get_best_experiment(metric='f1')
print(f"\nBest experiment: {best['name']}")
print(f"F1 Score: {best['metrics']['f1']:.4f}")

### Model Registry

**Purpose:** Centralized repository for model versions

In [ ]:
class ModelRegistry:
    """Model versioning and registry"""

    def __init__(self, storage_path='./models'):
        self.storage_path = storage_path
        self.registry = {}
        os.makedirs(storage_path, exist_ok=True)

    def register_model(self, name, model, version, metrics, metadata=None):
        """Register model version"""
        # Save model
        model_path = os.path.join(self.storage_path, f"{name}_v{version}.pkl")
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)

        # Register metadata
        if name not in self.registry:
            self.registry[name] = []

        version_info = {
            'version': version,
            'path': model_path,
            'metrics': metrics,
            'metadata': metadata or {},
            'timestamp': datetime.now().isoformat(),
            'stage': 'staging'  # staging, production, archived
        }

        self.registry[name].append(version_info)
        print(f"Model '{name}' v{version} registered")

    def promote_to_production(self, name, version):
        """Promote model to production"""
        for v in self.registry[name]:
            if v['version'] == version:
                v['stage'] = 'production'
                print(f"Model '{name}' v{version} promoted to production")
                return
        print(f"Version {version} not found")

    def load_model(self, name, version=None, stage='production'):
        """Load model by version or stage"""
        if name not in self.registry:
            raise ValueError(f"Model '{name}' not found")

        if version:
            for v in self.registry[name]:
                if v['version'] == version:
                    with open(v['path'], 'rb') as f:
                        return pickle.load(f)
        else:
            # Get model by stage
            for v in reversed(self.registry[name]):
                if v['stage'] == stage:
                    with open(v['path'], 'rb') as f:
                        return pickle.load(f)

        raise ValueError(f"No model found for stage '{stage}'")

    def list_models(self, name):
        """List all versions of a model"""
        if name not in self.registry:
            return pd.DataFrame()
        return pd.DataFrame(self.registry[name])

# Example usage
registry = ModelRegistry()

# Register models
registry.register_model(
    name='fraud_detector',
    model=model1,
    version='1.0',
    metrics={'accuracy': 0.85, 'f1': 0.82}
)

registry.register_model(
    name='fraud_detector',
    model=model2,
    version='2.0',
    metrics={'accuracy': 0.88, 'f1': 0.86}
)

# Promote to production
registry.promote_to_production('fraud_detector', '2.0')

# List versions
print("\nModel versions:")
print(registry.list_models('fraud_detector'))

# Load production model
production_model = registry.load_model('fraud_detector', stage='production')
print(f"\nLoaded production model: {type(production_model).__name__}")

---
## 12.4 Deployment Strategies

**Common Patterns:**
1. **Batch Prediction:** Scheduled inference on batches
2. **Online/Real-time:** API endpoint for instant predictions
3. **Streaming:** Process data streams (Kafka, Kinesis)
4. **Edge Deployment:** On-device inference

### Deployment Strategies Comparison

| Strategy | Description | Use Case |
|----------|-------------|----------|
| **Blue-Green** | Two identical environments, switch traffic | Zero-downtime deployment |
| **Canary** | Gradual rollout to subset of users | Test new version safely |
| **A/B Testing** | Compare two models with different user groups | Business metric optimization |
| **Shadow** | Run new model in parallel, don't serve results | Validate before switching |
| **Rolling** | Gradually replace old instances | Resource-efficient |
| **Multi-armed Bandit** | Dynamically allocate traffic based on performance | Optimize exploration/exploitation |

In [ ]:
# Simple Model Serving Interface
class ModelServer:
    """Basic model serving with versioning"""

    def __init__(self):
        self.models = {}  # {version: model}
        self.active_version = None
        self.prediction_count = {}

    def load_model(self, version, model):
        """Load model version"""
        self.models[version] = model
        self.prediction_count[version] = 0
        print(f"Loaded model version {version}")

    def set_active_version(self, version):
        """Set active model version"""
        if version not in self.models:
            raise ValueError(f"Version {version} not loaded")
        self.active_version = version
        print(f"Active version set to {version}")

    def predict(self, X, version=None):
        """Make prediction"""
        version = version or self.active_version
        if version not in self.models:
            raise ValueError(f"Version {version} not available")

        prediction = self.models[version].predict(X)
        self.prediction_count[version] += len(X)
        return prediction

    def get_stats(self):
        """Get serving statistics"""
        return pd.DataFrame([
            {
                'version': v,
                'predictions': count,
                'active': v == self.active_version
            }
            for v, count in self.prediction_count.items()
        ])

# Example usage
server = ModelServer()
server.load_model('1.0', model1)
server.load_model('2.0', model2)
server.set_active_version('2.0')

# Make predictions
test_data = X_test.iloc[:5]
predictions = server.predict(test_data)
print(f"\nPredictions: {predictions}")

# View stats
print("\nServer statistics:")
print(server.get_stats())

### Canary Deployment

In [ ]:
class CanaryDeployment:
    """Canary deployment with traffic splitting"""

    def __init__(self, stable_model, canary_model, canary_traffic=0.1):
        self.stable_model = stable_model
        self.canary_model = canary_model
        self.canary_traffic = canary_traffic
        self.metrics = {'stable': [], 'canary': []}

    def predict(self, X):
        """Route prediction to stable or canary"""
        if np.random.random() < self.canary_traffic:
            return self.canary_model.predict(X), 'canary'
        else:
            return self.stable_model.predict(X), 'stable'

    def log_metric(self, version, metric_value):
        """Log prediction metric"""
        self.metrics[version].append(metric_value)

    def should_promote_canary(self, threshold=0.95):
        """Decide if canary should be promoted"""
        if len(self.metrics['canary']) < 10:  # need enough samples
            return False

        canary_avg = np.mean(self.metrics['canary'])
        stable_avg = np.mean(self.metrics['stable'])

        return canary_avg >= stable_avg * threshold

    def increase_canary_traffic(self, increment=0.1):
        """Gradually increase canary traffic"""
        self.canary_traffic = min(1.0, self.canary_traffic + increment)
        print(f"Canary traffic increased to {self.canary_traffic:.1%}")

# Example
canary = CanaryDeployment(model1, model2, canary_traffic=0.1)

# Simulate predictions
for i in range(50):
    sample = X_test.iloc[[i % len(X_test)]]
    pred, version = canary.predict(sample)

    # Simulate metric (accuracy)
    metric = np.random.random()
    canary.log_metric(version, metric)

# Check if should promote
if canary.should_promote_canary():
    print("Canary performing well, increasing traffic")
    canary.increase_canary_traffic(0.2)
else:
    print("Canary not ready for promotion")

print(f"\nStable avg metric: {np.mean(canary.metrics['stable']):.3f}")
print(f"Canary avg metric: {np.mean(canary.metrics['canary']):.3f}")

---
## 12.5 Model Monitoring

**What to Monitor:**
1. **Prediction Quality:** Accuracy, precision, recall
2. **Data Drift:** Input distribution changes
3. **Concept Drift:** Target relationship changes
4. **Performance:** Latency, throughput
5. **System Health:** CPU, memory, errors

Great — this is a **production ML / MLOps topic** (very important for MLE interviews). Let’s break each item with **clear intuition + real examples + how to implement** 👇

---

### 📌 Model Monitoring — Why it matters

👉 After deployment, models **don’t stay correct forever**
Because:

* Data changes
* User behavior changes
* System conditions change

👉 Monitoring ensures:

```text
Model stays accurate + reliable in production
```

---

### 🧠 1. Prediction Quality (Model Performance)

#### What to monitor:

* Accuracy
* Precision / Recall
* F1-score
* AUC

---

### 📊 Example

Fraud model:

```text
Before: Recall = 0.85  
After 2 months: Recall = 0.55 ❌
```

👉 Model is degrading

---

### ⚙️ How to implement:

* Compare predictions vs actual labels (delayed feedback)
* Use dashboards (Prometheus, Grafana)

---

### 🔄 2. Data Drift (Input Distribution Change)

👉 Input data distribution changes over time

---

### 📊 Example

Training data:

```text
Age: 20–40
```

Production data:

```text
Age: 50–70 ❌
```

---

### 📉 Detection Methods

* PSI (Population Stability Index)
* KL Divergence
* Statistical tests

---

### ⚙️ Code Example (Simple)

```python
import numpy as np

np.mean(train_data), np.mean(prod_data)
```

👉 Compare distributions

---

### 🔥 3. Concept Drift (Most Critical 🚨)

👉 Relationship between **input → target changes**

---

### 📊 Example

Before:

```text
High income → low default
```

After recession:

```text
High income → high default ❌
```

---

### 🧠 Key Point

* Data looks same
* But model predictions become wrong

---

### ⚙️ Detection

* Drop in model performance
* Error rate monitoring

---

### ⚡ 4. Performance Monitoring (System)

👉 Check model serving performance

---

### Metrics:

* Latency (response time)
* Throughput (requests/sec)

---

### 📊 Example

```text
Latency > 500ms ❌ → bad user experience
```

---

### 🖥️ 5. System Health Monitoring

👉 Infrastructure-level checks

---

### Metrics:

* CPU usage
* Memory usage
* Error rate
* Logs

---

### 📊 Example

```text
Memory spike → model crashes ❌
```

---

### ⚔️ Data Drift vs Concept Drift

| Feature        | Data Drift   | Concept Drift |
| -------------- | ------------ | ------------- |
| What changes   | Input data   | Relationship  |
| Detectable via | Distribution | Performance   |
| Hardness       | Easy         | Hard 🔥       |

---

### 🚀 Real-world Monitoring Stack

* Logging → Kafka
* Metrics → Prometheus
* Dashboard → Grafana
* Alerts → PagerDuty

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If interviewer asks:

**“How do you monitor ML models?”**

Answer:

1. Monitor performance metrics
2. Detect data drift
3. Detect concept drift
4. Monitor latency
5. Monitor system health

---

### ⚡ Pro Strategy (MLE Level)

```text
1. Track input distributions
2. Track prediction distributions
3. Monitor performance metrics
4. Set alert thresholds
5. Retrain when drift detected
```

---

### 💡 One-liner to Remember

👉 **Model Monitoring = Accuracy + Drift + System + Performance**

---


### 📌 Population Stability Index (PSI)

PSI is a metric used to **detect data drift** by comparing the **distribution of a feature in training vs production**.

---

### 🧠 Intuition

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img0_fa51b6b0.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img1_508c8ce5.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img2_816a15fa.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img3_49db8f86.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img4_2740ea3b.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/sprashant433/MLOps/main/images/Part12_MLOps_Production_ML_cell17_img5_b9996591.png" alt="Image" style="max-width:90%;height:auto;" />
</div>

👉 Think like this:

> “Has the distribution of my data changed over time?”

---

### 🔢 PSI Formula
$$
PSI = \sum_{i=1}^{n} \left(A_i - E_i\right) \cdot \ln\left(\frac{A_i}{E_i}\right)
$$
Where:

* $A_i$ = Actual (production) proportion in bin $i$
* $E_i$ = Expected (training) proportion in bin $i$
* $n$ = number of bins

---

### 📊 Step-by-Step Example

### 🔹 Step 1: Create bins

Suppose feature = **Age**

| Bin | Range |
| --- | ----- |
| B1  | 20–30 |
| B2  | 30–40 |
| B3  | 40–50 |

---

### 🔹 Step 2: Calculate distributions

#### Training (Expected)

| Bin | %   |
| --- | --- |
| B1  | 0.5 |
| B2  | 0.3 |
| B3  | 0.2 |

---

#### Production (Actual)

| Bin | %   |
| --- | --- |
| B1  | 0.3 |
| B2  | 0.4 |
| B3  | 0.3 |

---

### 🔹 Step 3: Apply formula

For each bin:

#### B1:
$$
(0.3 - 0.5) \cdot \ln(0.3 / 0.5) \approx 0.102
$$
#### B2:
$$
(0.4 - 0.3) \cdot \ln(0.4 / 0.3) \approx 0.029
$$
#### B3:
$$
(0.3 - 0.2) \cdot \ln(0.3 / 0.2) \approx 0.041
$$
---

### 🔹 Final PSI

```text
PSI \approx 0.102 + 0.029 + 0.041 = 0.172
```

---

### 📉 Interpretation

| PSI Value  | Meaning              |
| ---------- | -------------------- |
| < 0.1      | No drift ✅           |
| 0.1 – 0.25 | Moderate drift ⚠️    |
| > 0.25     | Significant drift 🚨 |

👉 Here:

```text
0.171 → Moderate drift
```

---

### 🚀 Python Implementation

```python
import numpy as np
import pandas as pd

def calculate_psi(expected, actual, bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]
    
    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)
    
    psi = np.sum((actual_perc - expected_perc) *
                 np.log((actual_perc + 1e-6) / (expected_perc + 1e-6)))
    
    return psi
```

---

### ⚔️ PSI vs Other Drift Metrics

| Metric        | Use                          |
| ------------- | ---------------------------- |
| PSI           | Feature drift (most used 🔥) |
| KS test       | Distribution difference      |
| KL divergence | Information theory           |

---

### 🧠 Why PSI is Popular

* Simple
* Interpretable
* Works well in production

---

### ⚠️ Limitations

* Depends on binning
* Not sensitive to all changes
* Works best for numerical features

---

### 🚀 Real-world Use Case

* Credit scoring
* Fraud detection
* Monitoring feature drift

---

### 🧠 Interview Insight (VERY IMPORTANT)

👉 If interviewer asks:

**“How do you detect data drift?”**

Answer:

* PSI (most common)
* KS test
* Compare distributions

---

### ⚡ Pro Tip (MLE Level)

* Monitor PSI for **each feature**
* Set alert threshold:

```text
PSI > 0.2 → trigger retraining
```

---

### 💡 One-liner to Remember

👉 **PSI = measure of distribution shift between training and production**

---


In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt

class ModelMonitor:
    """Monitor model performance and data drift"""

    def __init__(self, reference_data):
        self.reference_data = reference_data
        self.predictions = []
        self.actuals = []
        self.timestamps = []

    def log_prediction(self, prediction, actual=None, timestamp=None):
        """Log prediction and actual"""
        self.predictions.append(prediction)
        if actual is not None:
            self.actuals.append(actual)
        self.timestamps.append(timestamp or datetime.now())

    def calculate_accuracy(self, window=100):
        """Calculate recent accuracy"""
        if len(self.actuals) < window:
            return None

        recent_preds = self.predictions[-window:]
        recent_actuals = self.actuals[-window:]
        return accuracy_score(recent_actuals, recent_preds)

    def detect_data_drift(self, current_data, feature, threshold=0.05):
        """Detect data drift using KS test"""
        reference = self.reference_data[feature].values
        current = current_data[feature].values

        statistic, p_value = ks_2samp(reference, current)

        is_drift = p_value < threshold
        return is_drift, statistic, p_value

    def check_all_features(self, current_data, threshold=0.05):
        """Check drift for all features"""
        results = []

        for feature in self.reference_data.columns:
            if feature not in current_data.columns:
                continue

            is_drift, statistic, p_value = self.detect_data_drift(
                current_data, feature, threshold
            )

            results.append({
                'feature': feature,
                'drift_detected': is_drift,
                'ks_statistic': statistic,
                'p_value': p_value
            })

        return pd.DataFrame(results)

# Example usage
monitor = ModelMonitor(X_train)

# Simulate predictions
for i in range(len(X_test)):
    pred = model2.predict(X_test.iloc[[i]])[0]
    actual = y_test.iloc[i]
    monitor.log_prediction(pred, actual)

# Check accuracy
accuracy = monitor.calculate_accuracy(window=20)
print(f"Recent accuracy: {accuracy:.3f}")

# Check for data drift
# Simulate drifted data
X_drifted = X_test.copy()
X_drifted['feature1'] = X_drifted['feature1'] + 2  # shift distribution

drift_results = monitor.check_all_features(X_drifted)
print("\nData drift detection:")
print(drift_results)

# Visualize drift
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, feature in enumerate(['feature1', 'feature2', 'feature3']):
    axes[i].hist(X_train[feature], bins=20, alpha=0.5, label='Training (Reference)')
    axes[i].hist(X_drifted[feature], bins=20, alpha=0.5, label='Current')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

    drift_info = drift_results[drift_results['feature'] == feature].iloc[0]
    axes[i].set_title(f"{feature}\nDrift: {drift_info['drift_detected']}")

plt.tight_layout()
plt.show()

---
## 12.6 CI/CD for ML

**Continuous Integration:**
- Code testing
- Data validation
- Model testing

**Continuous Deployment:**
- Automated retraining
- Model deployment
- Rollback mechanism

### Model Testing

**Test Types:**
1. **Unit Tests:** Individual functions
2. **Integration Tests:** Pipeline components
3. **Model Tests:** Performance benchmarks
4. **Data Tests:** Schema and quality

In [ ]:
class ModelTester:
    """Automated model testing"""

    def __init__(self, model, X_test, y_test):
        self.model = model
        self.X_test = X_test
        self.y_test = y_test
        self.test_results = []

    def test_accuracy_threshold(self, min_accuracy=0.7):
        """Test if accuracy meets threshold"""
        y_pred = self.model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)

        passed = accuracy >= min_accuracy
        self.test_results.append({
            'test': 'accuracy_threshold',
            'passed': passed,
            'value': accuracy,
            'threshold': min_accuracy
        })
        return passed

    def test_prediction_consistency(self, tolerance=0.01):
        """Test if predictions are consistent"""
        pred1 = self.model.predict(self.X_test)
        pred2 = self.model.predict(self.X_test)

        consistency = np.mean(pred1 == pred2)
        passed = consistency >= (1 - tolerance)

        self.test_results.append({
            'test': 'prediction_consistency',
            'passed': passed,
            'value': consistency,
            'threshold': 1 - tolerance
        })
        return passed

    def test_inference_time(self, max_time_ms=100):
        """Test inference time"""
        import time

        start = time.time()
        self.model.predict(self.X_test.iloc[:100])
        elapsed = (time.time() - start) * 1000  # ms

        passed = elapsed < max_time_ms
        self.test_results.append({
            'test': 'inference_time',
            'passed': passed,
            'value': elapsed,
            'threshold': max_time_ms
        })
        return passed

    def test_no_bias(self, sensitive_feature=None, threshold=0.1):
        """Test for bias (if sensitive feature provided)"""
        if sensitive_feature is None:
            return True

        y_pred = self.model.predict(self.X_test)

        # Split by sensitive feature
        group0 = self.y_test[sensitive_feature == 0]
        group1 = self.y_test[sensitive_feature == 1]

        pred0 = y_pred[sensitive_feature == 0]
        pred1 = y_pred[sensitive_feature == 1]

        acc0 = accuracy_score(group0, pred0)
        acc1 = accuracy_score(group1, pred1)

        bias = abs(acc0 - acc1)
        passed = bias < threshold

        self.test_results.append({
            'test': 'no_bias',
            'passed': passed,
            'value': bias,
            'threshold': threshold
        })
        return passed

    def run_all_tests(self):
        """Run all tests"""
        self.test_results = []

        self.test_accuracy_threshold()
        self.test_prediction_consistency()
        self.test_inference_time()

        results_df = pd.DataFrame(self.test_results)
        all_passed = results_df['passed'].all()

        return all_passed, results_df

# Example usage
tester = ModelTester(model2, X_test, y_test)
all_passed, results = tester.run_all_tests()

print("Model Test Results:")
print(results)
print(f"\nAll tests passed: {all_passed}")

---
### MLOps Tools & Platforms

**Experiment Tracking:**
- MLflow
- Weights & Biases
- Neptune.ai
- Comet.ml

**Model Serving:**
- TensorFlow Serving
- TorchServe
- Seldon Core
- BentoML
- KServe

**Complete Platforms:**
- Kubeflow
- MLflow
- AWS SageMaker
- Google Vertex AI
- Azure ML
- Databricks

**Orchestration:**
- Airflow
- Prefect
- Kubeflow Pipelines
- Metaflow

**Monitoring:**
- Evidently AI
- WhyLabs
- Arize AI
- Fiddler

**Data Versioning:**
- DVC (Data Version Control)
- Pachyderm
- LakeFS

---
### Quick Reference Guide

**MLOps Maturity Levels:**

| Level | Description | Characteristics |
|-------|-------------|----------------|
| **Level 0** | Manual | Notebooks, manual deployment |
| **Level 1** | ML Pipeline | Automated training pipeline |
| **Level 2** | CI/CD Pipeline | Automated testing and deployment |
| **Level 3** | Full MLOps | Monitoring, automated retraining |

**Best Practices:**

1. **Version Everything:**
   - Code (Git)
   - Data (DVC)
   - Models (Model Registry)
   - Environment (Docker)

2. **Automate:**
   - Testing
   - Training
   - Deployment
   - Monitoring

3. **Monitor:**
   - Model performance
   - Data drift
   - System health
   - Business metrics

4. **Document:**
   - Model cards
   - Data sheets
   - Experiment logs
   - Deployment procedures

5. **Test:**
   - Unit tests
   - Integration tests
   - Model performance tests
   - A/B tests

**Key Metrics:**
- **Model Metrics:** Accuracy, precision, recall, F1
- **System Metrics:** Latency, throughput, uptime
- **Data Metrics:** Drift score, data quality
- **Business Metrics:** ROI, user satisfaction

**Common Challenges:**
- Training-serving skew
- Data drift
- Concept drift
- Model decay
- Resource constraints
- Debugging production models
- Reproducibility

### Complete MLOps Automation with Google Cloud Build

Yes! We can absolutely automate the **entire MLOps pipeline** using Google Cloud Build. Cloud Build is perfect for this because it integrates seamlessly with GCP services (Vertex AI, Cloud Storage, Artifact Registry) and can orchestrate complex workflows.

Let me show you how to automate everything end-to-end.

---

### 🏗️ Architecture Overview

**Layman Terms**:
Cloud Build acts as your automated factory manager - whenever you make changes, it automatically runs all steps (data validation → training → testing → deployment) without you lifting a finger.

**Technical Terms**:
Cloud Build provides serverless CI/CD with native GCP integration, parallel builds, container-based execution, and built-in secret management.

---

### 📋 Complete Setup

#### **Step 1: Project Structure**

```
ml-project/
├── cloudbuild.yaml              # Main Cloud Build configuration
├── cloudbuild-trigger.yaml      # Trigger configuration
├── cloudbuild-deploy.yaml       # Deployment configuration
├── data/
│   ├── raw/
│   └── processed/
├── models/
├── scripts/
│   ├── validate_data.py
│   ├── preprocess.py
│   ├── train.py
│   ├── evaluate.py
│   ├── register_model.py
│   ├── deploy_model.py
│   └── monitor_setup.py
├── tests/
│   ├── unit/
│   ├── integration/
│   └── performance/
├── config/
│   ├── training_config.yaml
│   └── deployment_config.yaml
├── Dockerfile
├── requirements.txt
├── dvc.yaml
├── params.yaml
└── README.md
```

---

#### **Step 2: Main Cloud Build Configuration**

Create `cloudbuild.yaml`:

```yaml
### Complete MLOps Pipeline with Cloud Build
### This automates: Data Validation → Training → Testing → Registration → Deployment → Monitoring

options:
  machineType: 'N1_HIGHCPU_8'
  logging: CLOUD_LOGGING_ONLY
  
timeout: '7200s'  # 2 hours

substitutions:
  _PROJECT_ID: 'your-project-id'
  _REGION: 'us-central1'
  _BUCKET_NAME: 'your-ml-bucket'
  _MLFLOW_TRACKING_URI: 'http://your-mlflow-server:5000'
  _MODEL_NAME: 'customer-churn-predictor'
  _ARTIFACT_REGISTRY: 'us-central1-docker.pkg.dev/your-project/ml-models'

### Define multiple build steps
steps:

  # =============================================================================
  # STAGE 1: ENVIRONMENT SETUP
  # =============================================================================
  
  - name: 'gcr.io/cloud-builders/gcloud'
    id: 'setup-environment'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        echo "Setting up environment variables..."
        echo "Project: ${_PROJECT_ID}"
        echo "Region: ${_REGION}"
        echo "Build ID: $BUILD_ID"
        gcloud config set project ${_PROJECT_ID}

  # =============================================================================
  # STAGE 2: DATA VERSIONING WITH DVC
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'setup-dvc'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install dvc[gs] --quiet
        dvc remote add -d storage gs://${_BUCKET_NAME}/dvc-storage
        dvc remote modify storage auth gcloud
    waitFor: ['setup-environment']

  - name: 'python:3.9'
    id: 'pull-data'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        dvc pull
        echo "Data pulled successfully"
        ls -lh data/raw/
    waitFor: ['setup-dvc']

  # =============================================================================
  # STAGE 3: DATA VALIDATION
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'validate-data'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install pandas great-expectations --quiet
        python scripts/validate_data.py
        
        if [ $? -eq 0 ]; then
          echo "✅ Data validation passed"
        else
          echo "❌ Data validation failed"
          exit 1
        fi
    waitFor: ['pull-data']

  - name: 'python:3.9'
    id: 'data-quality-report'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install pandas-profiling ydata-profiling --quiet
        python scripts/generate_data_report.py
        gsutil cp data_quality_report.html gs://${_BUCKET_NAME}/reports/$BUILD_ID/
    waitFor: ['validate-data']

  # =============================================================================
  # STAGE 4: PREPROCESSING
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'preprocess-data'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install -r requirements.txt --quiet
        python scripts/preprocess.py
        
        # Version processed data with DVC
        dvc add data/processed/train.csv
        dvc add data/processed/test.csv
        dvc push
    waitFor: ['validate-data']

  # =============================================================================
  # STAGE 5: MODEL TRAINING WITH MLFLOW
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'train-model'
    entrypoint: 'bash'
    env:
      - 'MLFLOW_TRACKING_URI=${_MLFLOW_TRACKING_URI}'
      - 'MLFLOW_EXPERIMENT_NAME=customer-churn-cloud-build'
      - 'GCP_PROJECT=${_PROJECT_ID}'
    secretEnv: ['WANDB_API_KEY']
    args:
      - '-c'
      - |
        pip install -r requirements.txt --quiet
        
        echo "Starting model training..."
        python scripts/train.py \
          --data-path data/processed/train.csv \
          --output-path models/ \
          --experiment-name customer-churn-cloud-build \
          --run-name build-$BUILD_ID
        
        if [ $? -eq 0 ]; then
          echo "✅ Model training completed"
        else
          echo "❌ Model training failed"
          exit 1
        fi
    waitFor: ['preprocess-data']

  # =============================================================================
  # STAGE 6: MODEL EVALUATION
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'evaluate-model'
    entrypoint: 'bash'
    env:
      - 'MLFLOW_TRACKING_URI=${_MLFLOW_TRACKING_URI}'
    args:
      - '-c'
      - |
        pip install -r requirements.txt --quiet
        
        python scripts/evaluate.py \
          --model-path models/model.pkl \
          --test-data data/processed/test.csv \
          --metrics-output metrics/evaluation.json
        
        # Check if model meets quality threshold
        python scripts/check_model_quality.py \
          --metrics-file metrics/evaluation.json \
          --min-accuracy 0.85
        
        if [ $? -eq 0 ]; then
          echo "✅ Model quality checks passed"
        else
          echo "❌ Model quality below threshold"
          exit 1
        fi
    waitFor: ['train-model']

  # =============================================================================
  # STAGE 7: UNIT TESTS
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'run-unit-tests'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install -r requirements.txt pytest pytest-cov --quiet
        
        pytest tests/unit/ -v --cov=scripts --cov-report=html --cov-report=term
        
        # Upload coverage report
        gsutil cp -r htmlcov gs://${_BUCKET_NAME}/test-reports/$BUILD_ID/coverage/
    waitFor: ['evaluate-model']

  # =============================================================================
  # STAGE 8: INTEGRATION TESTS
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'run-integration-tests'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install -r requirements.txt pytest --quiet
        
        pytest tests/integration/ -v
    waitFor: ['run-unit-tests']

  # =============================================================================
  # STAGE 9: MODEL REGISTRATION IN MLFLOW
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'register-model-mlflow'
    entrypoint: 'bash'
    env:
      - 'MLFLOW_TRACKING_URI=${_MLFLOW_TRACKING_URI}'
    args:
      - '-c'
      - |
        pip install mlflow google-cloud-storage --quiet
        
        python scripts/register_model.py \
          --model-name ${_MODEL_NAME} \
          --model-path models/model.pkl \
          --build-id $BUILD_ID \
          --metrics-file metrics/evaluation.json
        
        echo "✅ Model registered in MLflow"
    waitFor: ['run-integration-tests']

  # =============================================================================
  # STAGE 10: BUILD SERVING CONTAINER
  # =============================================================================
  
  - name: 'gcr.io/cloud-builders/docker'
    id: 'build-serving-container'
    args:
      - 'build'
      - '-t'
      - '${_ARTIFACT_REGISTRY}/${_MODEL_NAME}:$BUILD_ID'
      - '-t'
      - '${_ARTIFACT_REGISTRY}/${_MODEL_NAME}:latest'
      - '-f'
      - 'Dockerfile.serving'
      - '.'
    waitFor: ['register-model-mlflow']

  - name: 'gcr.io/cloud-builders/docker'
    id: 'push-serving-container'
    args:
      - 'push'
      - '--all-tags'
      - '${_ARTIFACT_REGISTRY}/${_MODEL_NAME}'
    waitFor: ['build-serving-container']

  # =============================================================================
  # STAGE 11: UPLOAD MODEL TO VERTEX AI
  # =============================================================================
  
  - name: 'gcr.io/google.com/cloudsdktool/cloud-sdk'
    id: 'upload-to-vertex-ai'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        python scripts/upload_to_vertex.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --model-name ${_MODEL_NAME} \
          --container-image ${_ARTIFACT_REGISTRY}/${_MODEL_NAME}:$BUILD_ID \
          --build-id $BUILD_ID
        
        echo "✅ Model uploaded to Vertex AI Model Registry"
    waitFor: ['push-serving-container']

  # =============================================================================
  # STAGE 12: DEPLOY TO STAGING
  # =============================================================================
  
  - name: 'gcr.io/google.com/cloudsdktool/cloud-sdk'
    id: 'deploy-to-staging'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        python scripts/deploy_model.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --model-name ${_MODEL_NAME} \
          --environment staging \
          --machine-type n1-standard-4 \
          --min-replicas 1 \
          --max-replicas 3
        
        # Save endpoint info
        echo "STAGING_ENDPOINT=$(cat endpoint_info.json | jq -r '.endpoint_id')" >> /workspace/endpoints.env
        
        echo "✅ Model deployed to staging"
    waitFor: ['upload-to-vertex-ai']

  # =============================================================================
  # STAGE 13: STAGING TESTS
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'test-staging-endpoint'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform requests --quiet
        
        source /workspace/endpoints.env
        
        python tests/integration/test_endpoint.py \
          --endpoint-id $STAGING_ENDPOINT \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION}
        
        if [ $? -eq 0 ]; then
          echo "✅ Staging endpoint tests passed"
        else
          echo "❌ Staging endpoint tests failed"
          exit 1
        fi
    waitFor: ['deploy-to-staging']

  # =============================================================================
  # STAGE 14: PERFORMANCE TESTS
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'performance-tests'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install locust google-cloud-aiplatform --quiet
        
        source /workspace/endpoints.env
        
        # Run load tests
        python tests/performance/load_test.py \
          --endpoint-id $STAGING_ENDPOINT \
          --users 50 \
          --spawn-rate 5 \
          --run-time 300s
        
        echo "✅ Performance tests completed"
    waitFor: ['test-staging-endpoint']

  # =============================================================================
  # STAGE 15: SETUP MONITORING
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'setup-monitoring'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform google-cloud-monitoring --quiet
        
        source /workspace/endpoints.env
        
        python scripts/monitor_setup.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --endpoint-id $STAGING_ENDPOINT \
          --alert-emails ml-team@company.com \
          --training-data gs://${_BUCKET_NAME}/data/processed/train.csv
        
        echo "✅ Monitoring configured"
    waitFor: ['deploy-to-staging']

  # =============================================================================
  # STAGE 16: GENERATE DEPLOYMENT REPORT
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'generate-report'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install jinja2 markdown --quiet
        
        python scripts/generate_deployment_report.py \
          --build-id $BUILD_ID \
          --metrics-file metrics/evaluation.json \
          --endpoint-file endpoint_info.json \
          --output reports/deployment_report_$BUILD_ID.html
        
        gsutil cp reports/deployment_report_$BUILD_ID.html gs://${_BUCKET_NAME}/reports/$BUILD_ID/
        
        echo "✅ Deployment report generated"
        echo "Report: gs://${_BUCKET_NAME}/reports/$BUILD_ID/deployment_report_$BUILD_ID.html"
    waitFor: ['performance-tests', 'setup-monitoring']

  # =============================================================================
  # STAGE 17: NOTIFY TEAM
  # =============================================================================
  
  - name: 'gcr.io/cloud-builders/gcloud'
    id: 'notify-team'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        source /workspace/endpoints.env
        
        # Send notification via Cloud Pub/Sub
        gcloud pubsub topics publish ml-deployments \
          --message="{
            \"build_id\": \"$BUILD_ID\",
            \"model_name\": \"${_MODEL_NAME}\",
            \"environment\": \"staging\",
            \"endpoint_id\": \"$STAGING_ENDPOINT\",
            \"status\": \"success\",
            \"report_url\": \"gs://${_BUCKET_NAME}/reports/$BUILD_ID/deployment_report_$BUILD_ID.html\"
          }"
        
        echo "✅ Team notified"
    waitFor: ['generate-report']

### Secret management for sensitive data
availableSecrets:
  secretManager:
    - versionName: projects/${_PROJECT_ID}/secrets/wandb-api-key/versions/latest
      env: 'WANDB_API_KEY'
    - versionName: projects/${_PROJECT_ID}/secrets/mlflow-tracking-token/versions/latest
      env: 'MLFLOW_TRACKING_TOKEN'

### Artifacts to save
artifacts:
  objects:
    location: 'gs://${_BUCKET_NAME}/builds/$BUILD_ID'
    paths:
      - 'models/**'
      - 'metrics/**'
      - 'reports/**'
      - 'endpoint_info.json'
```

---

#### **Step 3: Production Deployment Configuration**

Create `cloudbuild-deploy-production.yaml`:

```yaml
### Production Deployment Pipeline - Requires Manual Approval
### This should be triggered separately after staging validation

steps:

  # =============================================================================
  # STAGE 1: VALIDATE STAGING PERFORMANCE
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'validate-staging-metrics'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-monitoring google-cloud-aiplatform --quiet
        
        python scripts/validate_staging_metrics.py \
          --project-id ${_PROJECT_ID} \
          --staging-endpoint-id ${_STAGING_ENDPOINT_ID} \
          --min-uptime 0.999 \
          --max-latency-p95 100 \
          --min-throughput 100
        
        if [ $? -eq 0 ]; then
          echo "✅ Staging metrics validation passed"
        else
          echo "❌ Staging metrics below production standards"
          exit 1
        fi

  # =============================================================================
  # STAGE 2: SECURITY SCAN
  # =============================================================================
  
  - name: 'gcr.io/cloud-builders/gcloud'
    id: 'security-scan'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        # Scan container image for vulnerabilities
        gcloud artifacts docker images scan ${_ARTIFACT_REGISTRY}/${_MODEL_NAME}:${_BUILD_ID}
        
        # Wait for scan to complete
        gcloud artifacts docker images list-vulnerabilities \
          ${_ARTIFACT_REGISTRY}/${_MODEL_NAME}:${_BUILD_ID} \
          --format=json > vulnerabilities.json
        
        # Check for critical vulnerabilities
        python scripts/check_vulnerabilities.py --report vulnerabilities.json
    waitFor: ['validate-staging-metrics']

  # =============================================================================
  # STAGE 3: COMPLIANCE CHECK
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'compliance-check'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        python scripts/compliance_check.py \
          --model-name ${_MODEL_NAME} \
          --build-id ${_BUILD_ID} \
          --check-pii \
          --check-bias \
          --check-explainability
        
        echo "✅ Compliance checks passed"
    waitFor: ['security-scan']

  # =============================================================================
  # STAGE 4: BLUE-GREEN DEPLOYMENT SETUP
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'setup-blue-green'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        # Deploy new version alongside existing production
        python scripts/blue_green_deploy.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --model-name ${_MODEL_NAME} \
          --new-version ${_BUILD_ID} \
          --initial-traffic 0 \
          --machine-type n1-standard-8 \
          --min-replicas 3 \
          --max-replicas 10
        
        echo "GREEN_ENDPOINT=$(cat green_endpoint.json | jq -r '.endpoint_id')" >> /workspace/prod_endpoints.env
    waitFor: ['compliance-check']

  # =============================================================================
  # STAGE 5: CANARY TESTING (10% TRAFFIC)
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'canary-10-percent'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        source /workspace/prod_endpoints.env
        
        # Route 10% traffic to new version
        python scripts/update_traffic_split.py \
          --endpoint-id $GREEN_ENDPOINT \
          --new-version-traffic 10 \
          --old-version-traffic 90
        
        echo "Canary deployed with 10% traffic"
        echo "Monitoring for 15 minutes..."
        sleep 900  # Wait 15 minutes
        
        # Check error rates and latency
        python scripts/validate_canary_metrics.py \
          --endpoint-id $GREEN_ENDPOINT \
          --duration 15m \
          --max-error-rate 0.01
    waitFor: ['setup-blue-green']

  # =============================================================================
  # STAGE 6: GRADUAL ROLLOUT (50% TRAFFIC)
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'rollout-50-percent'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        source /workspace/prod_endpoints.env
        
        # Route 50% traffic to new version
        python scripts/update_traffic_split.py \
          --endpoint-id $GREEN_ENDPOINT \
          --new-version-traffic 50 \
          --old-version-traffic 50
        
        echo "50% traffic routed to new version"
        echo "Monitoring for 15 minutes..."
        sleep 900
        
        python scripts/validate_canary_metrics.py \
          --endpoint-id $GREEN_ENDPOINT \
          --duration 15m \
          --max-error-rate 0.01
    waitFor: ['canary-10-percent']

  # =============================================================================
  # STAGE 7: FULL DEPLOYMENT (100% TRAFFIC)
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'full-deployment'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        source /workspace/prod_endpoints.env
        
        # Route 100% traffic to new version
        python scripts/update_traffic_split.py \
          --endpoint-id $GREEN_ENDPOINT \
          --new-version-traffic 100 \
          --old-version-traffic 0
        
        echo "✅ 100% traffic routed to new version"
    waitFor: ['rollout-50-percent']

  # =============================================================================
  # STAGE 8: UPDATE MLFLOW MODEL REGISTRY
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'promote-to-production'
    entrypoint: 'bash'
    env:
      - 'MLFLOW_TRACKING_URI=${_MLFLOW_TRACKING_URI}'
    args:
      - '-c'
      - |
        pip install mlflow --quiet
        
        python scripts/promote_model.py \
          --model-name ${_MODEL_NAME} \
          --version ${_BUILD_ID} \
          --stage Production
        
        echo "✅ Model promoted to Production in MLflow"
    waitFor: ['full-deployment']

  # =============================================================================
  # STAGE 9: ARCHIVE OLD VERSION
  # =============================================================================
  
  - name: 'python:3.9'
    id: 'archive-old-version'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform --quiet
        
        python scripts/archive_old_deployment.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --keep-last-n 3
        
        echo "✅ Old versions archived"
    waitFor: ['promote-to-production']

  # =============================================================================
  # STAGE 10: PRODUCTION NOTIFICATION
  # =============================================================================
  
  - name: 'gcr.io/cloud-builders/gcloud'
    id: 'notify-production-deployment'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        source /workspace/prod_endpoints.env
        
        gcloud pubsub topics publish ml-deployments \
          --message="{
            \"build_id\": \"${_BUILD_ID}\",
            \"model_name\": \"${_MODEL_NAME}\",
            \"environment\": \"production\",
            \"deployment_type\": \"blue-green\",
            \"endpoint_id\": \"$GREEN_ENDPOINT\",
            \"status\": \"success\"
          }"
        
        echo "✅ Production deployment completed and team notified"
    waitFor: ['archive-old-version']

substitutions:
  _PROJECT_ID: 'your-project-id'
  _REGION: 'us-central1'
  _MODEL_NAME: 'customer-churn-predictor'
  _BUILD_ID: 'latest'
  _STAGING_ENDPOINT_ID: 'staging-endpoint-id'
  _ARTIFACT_REGISTRY: 'us-central1-docker.pkg.dev/your-project/ml-models'
  _MLFLOW_TRACKING_URI: 'http://your-mlflow-server:5000'

timeout: '3600s'
```

---

#### **Step 4: Automated Triggers Configuration**

Create `cloudbuild-triggers.tf` (Terraform):

```hcl
### Terraform configuration for Cloud Build triggers

resource "google_cloudbuild_trigger" "ml_pipeline_dev" {
  name        = "ml-pipeline-dev"
  description = "Trigger ML pipeline on dev branch commits"
  
  github {
    owner = "your-github-org"
    name  = "ml-project"
    
    push {
      branch = "^dev$"
    }
  }
  
  filename = "cloudbuild.yaml"
  
  substitutions = {
    _ENVIRONMENT = "development"
    _PROJECT_ID  = var.project_id
  }
}

resource "google_cloudbuild_trigger" "ml_pipeline_staging" {
  name        = "ml-pipeline-staging"
  description = "Trigger ML pipeline on main branch commits"
  
  github {
    owner = "your-github-org"
    name  = "ml-project"
    
    push {
      branch = "^main$"
    }
  }
  
  filename = "cloudbuild.yaml"
  
  substitutions = {
    _ENVIRONMENT = "staging"
    _PROJECT_ID  = var.project_id
  }
  
  # Auto-deploy to staging
  included_files = [
    "scripts/**",
    "models/**",
    "data/**",
    "requirements.txt"
  ]
}

resource "google_cloudbuild_trigger" "ml_pipeline_production" {
  name        = "ml-pipeline-production"
  description = "Deploy to production (requires approval)"
  
  github {
    owner = "your-github-org"
    name  = "ml-project"
    
    # Trigger on release tags
    tag = "^v[0-9]+\\.[0-9]+\\.[0-9]+$"
  }
  
  filename = "cloudbuild-deploy-production.yaml"
  
  substitutions = {
    _ENVIRONMENT = "production"
    _PROJECT_ID  = var.project_id
  }
  
  # Require approval
  approval_config {
    approval_required = true
  }
}

### Scheduled trigger for retraining
resource "google_cloudbuild_trigger" "ml_scheduled_retrain" {
  name        = "ml-scheduled-retrain"
  description = "Weekly model retraining"
  
  # Trigger every Sunday at 2 AM
  pubsub_config {
    topic = google_pubsub_topic.ml_retrain_schedule.id
  }
  
  filename = "cloudbuild.yaml"
  
  substitutions = {
    _TRIGGER_TYPE = "scheduled"
    _PROJECT_ID   = var.project_id
  }
}

resource "google_pubsub_topic" "ml_retrain_schedule" {
  name = "ml-retrain-schedule"
}

resource "google_cloud_scheduler_job" "weekly_retrain" {
  name        = "weekly-model-retrain"
  description = "Trigger weekly model retraining"
  schedule    = "0 2 * * 0"  # Every Sunday at 2 AM
  time_zone   = "America/New_York"
  
  pubsub_target {
    topic_name = google_pubsub_topic.ml_retrain_schedule.id
    data       = base64encode("{\"trigger\": \"scheduled_retrain\"}")
  }
}
```

---

#### **Step 5: Supporting Python Scripts**

**scripts/train.py**:

```python
import argparse
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import pickle
import json
import os

def train_model(args):
    # Set MLflow tracking
    mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))
    mlflow.set_experiment(args.experiment_name)
    
    with mlflow.start_run(run_name=args.run_name):
        # Load data
        df = pd.read_csv(args.data_path)
        X = df.drop('churned', axis=1)
        y = df['churned']
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Log parameters
        params = {
            'n_estimators': 100,
            'max_depth': 10,
            'min_samples_split': 5,
            'random_state': 42
        }
        mlflow.log_params(params)
        
        # Train model
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        
        # Evaluate
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        
        mlflow.log_metric("train_accuracy", train_score)
        mlflow.log_metric("test_accuracy", test_score)
        
        # Save model locally
        os.makedirs(args.output_path, exist_ok=True)
        model_path = os.path.join(args.output_path, 'model.pkl')
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        
        # Log model to MLflow
        mlflow.sklearn.log_model(model, "model")
        
        print(f"✅ Model trained successfully")
        print(f"Train accuracy: {train_score:.4f}")
        print(f"Test accuracy: {test_score:.4f}")
        
        return model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-path', required=True)
    parser.add_argument('--output-path', required=True)
    parser.add_argument('--experiment-name', required=True)
    parser.add_argument('--run-name', required=True)
    
    args = parser.parse_args()
    train_model(args)
```

**scripts/deploy_model.py**:

```python
import argparse
from google.cloud import aiplatform
import json

def deploy_model(args):
    aiplatform.init(project=args.project_id, location=args.region)
    
    # Get the latest model
    models = aiplatform.Model.list(
        filter=f'display_name="{args.model_name}"',
        order_by="create_time desc"
    )
    
    if not models:
        raise ValueError(f"No model found with name {args.model_name}")
    
    model = models[0]
    
    # Create or get endpoint
    endpoint_name = f"{args.model_name}-{args.environment}"
    endpoints = aiplatform.Endpoint.list(
        filter=f'display_name="{endpoint_name}"'
    )
    
    if endpoints:
        endpoint = endpoints[0]
        print(f"Using existing endpoint: {endpoint.resource_name}")
    else:
        endpoint = aiplatform.Endpoint.create(
            display_name=endpoint_name,
            labels={"environment": args.environment}
        )
        print(f"Created new endpoint: {endpoint.resource_name}")
    
    # Deploy model
    deployed_model = model.deploy(
        endpoint=endpoint,
        deployed_model_display_name=f"{args.model_name}-{args.environment}-v1",
        machine_type=args.machine_type,
        min_replica_count=args.min_replicas,
        max_replica_count=args.max_replicas,
        traffic_percentage=100,
        sync=True
    )
    
    # Save endpoint info
    endpoint_info = {
        "endpoint_id": endpoint.name,
        "endpoint_name": endpoint.display_name,
        "model_id": model.name,
        "environment": args.environment,
        "resource_name": endpoint.resource_name
    }
    
    with open('endpoint_info.json', 'w') as f:
        json.dump(endpoint_info, f, indent=2)
    
    print(f"✅ Model deployed to {args.environment}")
    print(f"Endpoint ID: {endpoint.name}")
    
    return endpoint

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--project-id', required=True)
    parser.add_argument('--region', required=True)
    parser.add_argument('--model-name', required=True)
    parser.add_argument('--environment', required=True)
    parser.add_argument('--machine-type', default='n1-standard-4')
    parser.add_argument('--min-replicas', type=int, default=1)
    parser.add_argument('--max-replicas', type=int, default=3)
    
    args = parser.parse_args()
    deploy_model(args)
```

---

#### **Step 6: Manual Trigger via gcloud**

```bash
### Trigger the main pipeline manually
gcloud builds submit \
  --config=cloudbuild.yaml \
  --substitutions=_PROJECT_ID="your-project-id",_REGION="us-central1"

### Trigger production deployment
gcloud builds submit \
  --config=cloudbuild-deploy-production.yaml \
  --substitutions=_PROJECT_ID="your-project-id",_BUILD_ID="build-123"

### Check build status
gcloud builds list --limit=10

### View build logs
gcloud builds log <BUILD_ID> --stream
```

---

#### **Step 7: Rollback Configuration**

Create `cloudbuild-rollback.yaml`:

```yaml
### Emergency rollback pipeline

steps:
  - name: 'python:3.9'
    id: 'rollback-to-previous'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        pip install google-cloud-aiplatform mlflow --quiet
        
        python scripts/rollback_deployment.py \
          --project-id ${_PROJECT_ID} \
          --region ${_REGION} \
          --model-name ${_MODEL_NAME} \
          --target-version ${_ROLLBACK_VERSION}
        
        echo "✅ Rolled back to version ${_ROLLBACK_VERSION}"
  
  - name: 'gcr.io/cloud-builders/gcloud'
    id: 'notify-rollback'
    entrypoint: 'bash'
    args:
      - '-c'
      - |
        gcloud pubsub topics publish ml-deployments \
          --message="{
            \"action\": \"rollback\",
            \"model_name\": \"${_MODEL_NAME}\",
            \"from_version\": \"${_CURRENT_VERSION}\",
            \"to_version\": \"${_ROLLBACK_VERSION}\",
            \"reason\": \"${_ROLLBACK_REASON}\"
          }"

substitutions:
  _PROJECT_ID: 'your-project-id'
  _REGION: 'us-central1'
  _MODEL_NAME: 'customer-churn-predictor'
  _ROLLBACK_VERSION: 'previous'
  _CURRENT_VERSION: 'current'
  _ROLLBACK_REASON: 'emergency'
```

---

### 🎯 Complete Automation Workflow

Here's how everything works together with Cloud Build:

1. **Developer pushes code to GitHub** → Webhook triggers Cloud Build
2. **Cloud Build pulls data via DVC** → Ensures data versioning
3. **Data validation runs** → Quality gates
4. **Model training with MLflow** → Experiment tracking
5. **Tests execute** → Unit, integration, performance
6. **Container image builds** → For serving
7. **Model uploads to Vertex AI** → Registry
8. **Auto-deploy to staging** → Vertex AI endpoint
9. **Monitoring setup** → Drift detection
10. **Manual approval required** → For production
11. **Blue-green deployment** → Zero-downtime
12. **Gradual traffic shift** → 10% → 50% → 100%
13. **Notifications sent** → Slack/Email/Pub/Sub

---

### 🚀 Benefits of Cloud Build Automation

**Layman Benefits**:
- **One Click**: Push code, everything else happens automatically
- **Fast**: Parallel execution speeds up deployment
- **Safe**: Multiple validation gates prevent bad models
- **Reliable**: Consistent process every time
- **Transparent**: Full audit trail of every deployment

**Technical Benefits**:
- **Native GCP Integration**: Seamless with Vertex AI, GCS, Artifact Registry
- **Parallel Execution**: Multiple stages run simultaneously
- **Cost Effective**: Pay only for build time
- **Scalable**: Handles any size pipeline
- **Secure**: Managed secrets, IAM integration
- **Flexible**: Custom containers, any language